# Оракул Блейхенбахера по паддингу
Современные варианты этой атаки были относительно недавно обнаружены [The ROBOT Attack](https://robotattack.org/), но и оригинальная уязвимость всё ещё скрывается в некоторых старых библиотеках (кстати, похожая уязвимость была исправлена в [OpenSSL 3.0.8](https://gist.github.com/guidovranken/96e9f24759242d24655b5d1a7051b1ad) 7 февраля 2023).

## Базовый RSA
У примитива RSA ($c=Enc(m)=m^e\ mod\ N; m=Dec(c)=c^d\ mod\ N$) есть определенные недостатки:
1. Если $m_1=m_2$, то $c_1=c_2$, что является проблемой, так как позволяет получить информацию при атаке "человек посередине" (Man-in-the-Middle). (Например, мы можем узнать, что тот же самый человек использует систему в данный момент времени, поскольку он послал тот же самый зашифрованный пароль)
2. Если $e\cdot bitlen(m)\lt bitlen(N)$, то $m^e$ не переполняет модуль и может быть легко расшифрован взятием $e$-того корня $C$.

## PKCS#1 v1.5
Чтобы противодействовать этим проблемам, RSA (компания) создала спецификацию, которая стала [PKCS\#1 v1.5](https://tools.ietf.org/html/rfc2313) (Public Key Cryptography Standards)
Спецификация вводит форматирование для шифруемых (или подписываемых) блоков (по-сути, специальный паддинг).
Пусть длина модуля $N$ в октетах (байтах) равна $k$: $k=||N||$. Данные, которые мы хотим зашифровать или подписать - это $D$. Тогда шифруемый блок будет выглядеть следующим образом:

``` EB = 00 || BT || PS || 00 || D ```

где $BT$ принимает значения $00$ или $01$ для операций на закрытом ключе (создание подписи) и $02$ для операций на открытом ключе (шифрование).
Шифруемый блок преобразуется в целое число с использованием формата big-endian (первый октет EB - наиболее значимый)
$PS$ состоит из $k-3-||D||$ октетов. Их содержимое зависит от $BT$:
1. $BT=00$, все октеты в $PS$ принимают значение $00$
2. $BT=01$, все октеты в $PS$ принимают значение FF
3. $BT=02$, все октеты в $PS$ сгенерированы псевдослучайным образом и не равны $00$

Рекомендуется использовать $BT=01$ для подписи $BT=02$ для шифрования, потому что
1. При $BT=01$ с $D$ может быть снято форматирование вне зависимости от его содержимого,а при $BT=00$ первый байт $D$, если $D_0=00$, тоже будет снят, что порождает проблемы.
2. И $BT=01$ и $BT=02$ генерируют большие числа для шифрования и подписи, поэтому все атаки, зависящие от малого  $D$ не работают.

Вы могли заметить, что первый октет $EB$ - $00$. Он выбран таким образом, чтобы целое число, полученное из $EB$ всегда было меньше $N$.

## Атака Блейхенбахера на Оракул по Паддингу
Лучшее всего описал ситуацию д.н. Мэтью Грин в[What is the Random Oracle Model and why should you care? (Part 4)](https://blog.cryptographyengineering.com/2011/11/02/what-is-random-oracle-model-and-why/)(вольный перевод) :
>Так, я писал, что ни у кого с практическим складом ума не было претензий к PKCS. Одним ярким исключением был талантливый криптограф из Bell Labs по имени Дэниэль Блейхенбахер.
>
>У д.н. Блейхенбахера были серьёзные претензии к стандарту паддинга PKCS. Вообще-то, можно даже сказать, что он поставил себе цель всей жизни их уничтожить. Эта вендетта достигла апогея в 2006м году, когда он показал, как взламывать популярные имплементации подписи PKCS при помощи листка бумаги и карандаша, незадолго до того как въехал в штаб-квартиру RSA Data Security на минивэне, забитом фейерверками.
>
>(Так и быть, последней части на самом деле не было, но всё остальное - правда.)
>
>Блейхенбахер нанёс первый удар по PKCS на конфереции CRYPTO 1998. В на удивление читаемой статье он предложил первую практическую адаптивную атаку с выбранным закрытым текстом на "протоколы, использующие стандарт шифрования PKCS #1". Так как, как я уже упоминал, значимым протоколом, который попадал под это определение, был SSL, это было действительно важное событие."

Вы можете взять оригинальную статью здесь: [Chosen Ciphertext Attacks Against Protocols Based on the RSA Encryption Standard PKCS#1](http://archiv.infsec.ethz.ch/education/fs08/secsem/bleichenbacher98.pdf).

Представим, что мы послали случайное целое число $c$ на расшифровку. Какова вероятность, что снятие паддинга не приведет к ошибке? Первые 2 октета $EB$ должны быть $00 || 02$, а среди остальных октетов должен присутствовать хотя бы один $00$. Если $k$ равно 256, то $P=\frac{1}{256^2}\cdot(1-(\frac{255}{256})^{254})\approx \frac{1}{104031}$ Кажется, что вероятность слишком маленькая, но на самом деле, она применима на практике. Атака работает следующим образом:

Выбираем $c$, для которого мы хотим получить открытый текст $m=c^d\ mod\ N$

Для удобства определим значение $B=2^{8(k-2)},\ k=||N||$. Мы говорим, что шифротекст соответствует PKCS, если он расшифровывается в соответствующий PKCS открытый текст.

Мы выбираем целые числа $s$, вычисляем шифротексты $c'=cs^e\ mod\ N$ и отсылаем их оракулу. Если $c'$ проходит проверку на ошибку, то $2B \le ms\ mod\ N \lt 3B$. Собрав достаточно различных $s$ мы можем вычислить $m$. Обычно требуется примерно $2^{20}$ шифротекстов, но это число сильно варьируется.
Атаку можно разделить на три этапа:
1. Маскирование (блайндинг) шифротекста, вычисление $c_0$, который соответствует неизвестному $m_0$.
2. Поиск таких малых $s_i$, что $m_0s_i\ mod\ N$ соответствует PKCS. На каждый такой $s_i$ атакующий вычисляет интервалы, которые должны содержать $m_0$, используя ранее известную информацию.
3. Начинается, когда остается только один интервал. У атакующего достаточно информации о $m_0$, чтобы выбрать $s_i$, такое что $m_0s_i\ mod\ N$ будет соответствовать PKCS с большей вероятностью, чем выбранное случайно сообщение. Постепенно увеличивая $s_i$, сужается возможный интервал $m_0$, пока не останется единственно возможное значение.

(Вообще-то, когда интервал достаточно мал, просто перебрать возможные значения локально более эффективно, чем отправлять запросы оракулу)

## Полный алгоритм (из статьи [paper](http://archiv.infsec.ethz.ch/education/fs08/secsem/bleichenbacher98.pdf))
$M_i$ - это множество (закрытых) интервалов, которое вычисляется после нахождения успешного (прошедшего проверку) $s_i$. $m_0$ содержится в одном из интервалов $M_i$
### Шаг 1: Блайндинг.
Для целого числа $c$, выберите различные случайные целые числа $s_0$; после проверьте, обращаясь к оракулу, расшифровываются ли числа $c(s_0)^e\ mod\ N$ в соответствующие PKCS $m'$. Для первого успешного  $s_0$ вычисляем:
$$c_0\leftarrow c(s_0)^e\ mod\ N$$
$$M_0\leftarrow \{[2B,3B-1]\}$$
$$i\leftarrow 1.$$
### Шаг 2: Поиск сообщений, соответствующих PKCS
#### Шаг 2.a: Начало поиска.
Если $i=1$, то ищите наименьшее положительное целое $s_1\ge N/(3B)$, такое что шифртекст $c_0(s_1)^e\ mod\ N$ соответствует PKCS.
#### Шаг 2.b: Поиск при нескольких интервалах.
В противном случае если $i \gt 1$ и количество интервалов в $M_{i-1}$ как минимум 2, то ищите наименьшее целое $s_i\gt s_{i-1}$, такое что шифртекст $c_0(s_i)^e\ mod\ N$ соответствует PKCS.
#### Шаг 2.c: Поиск при одном интервале.
Если $M_{i-1}$ содержит лишь один интервал (например, $M_{i-1} = \{[a,b]\}$)), то перебирайте малые целые значения $r_i,s_i$, такие что
$$ r_i \ge 2\frac{bs_{i-1}-2B}{N} \tag{1}$$
и
$$\frac{2B+r_iN}{b}\le s_i \lt \frac{3B+r_iN}{a}, \tag{2}$$
пока шифртекст $c_0(s_i)^e\ mod\ N$ не будет соответствовать PKCS.

### Шаг 3: Сужение множества решений.
После того, как $s_i$ было найдено, множество $M_i$ может быть вычислено как
$$M_i \leftarrow \mathop{\bigcup}_{(a,b,r)} \Bigg \{ \Bigg [max \Bigg ( a, \Bigg \lceil \frac{2B+rN}{s_i} \Bigg \rceil \Bigg),min \Bigg( b, \Bigg \lfloor \frac{3B-1+rN}{s_i} \Bigg \rfloor \Bigg) \Bigg ] \Bigg \} \tag{3}$$
для всех $[a,b] \in M_{i-1}$ и $$\frac{as_i-3B+1}{N}\le r\le \frac{bs_i-2B}{N}$$
### Шаг 4: Вычисление решения.
Если $M_i$ содержит лишь один интервал длины 1 (например, $M_i=\{[a,a]\}$), то присвойте $m\leftarrow a(s_0)^{-1}\ mod\ N$, и верните $m$ как решение $m=c^d\ mod\ N$. В противном случае выставите $i \leftarrow i+1$ и перейдите на шаг 2.

### Замечания.
Шаг 1 может быть пропущен, если $c$ уже соответствует PKCS (например, когда c - зашифрованное сообщение). В этом случае, присвойте $s_0 \leftarrow 1$. Однако шаг 1 всегда необходим для вычисления подписи, даже если Вы не хотите получить "слепую" подпись.

На шаге 2.a, надо начинать с $s_1=\lceil N/(3B)\rceil$, потому что для меньших значений $m_0s_1$ никогда не будет соответствовать PKCS.

Мы используем условие (1), потому что за каждую итерацию хотим разделить оставшийся интервал примерно пополам. Также можно использовать перебор на последних стадиях атаки, когда пространство сообщений достаточно маленькое.

## Задание
Вам предоставлен сервер с оракулом Блейхенбахера по паддингу. Реализуйте атаку и дешифруйте шифртекст (он соответствует PKCS). В первом случае вам предоставлена локальная версия сервера, чтобы ускорить тестирование и разработку. Попытайтесь использовать функцию **get_pkcs_conforming_mult**, она ускоряет взаимодействие с сервером, что будет особо важно, когда вы будете посылать реальные запросы. Ей просто нужно предоставить список значений

In [3]:
from Crypto.PublicKey import RSA
from Crypto.Cipher import PKCS1_v1_5
from Crypto.Util.number import bytes_to_long, long_to_bytes, inverse
from Crypto.Random import get_random_bytes
class LocalVulnServerClient:
    def __init__(self,show=True):
        """Initialization, generate key"""
        self.key=RSA.generate(2048)
        self.cipher=PKCS1_v1_5.new(self.key)
        self.flag_message="FLAG FOR TESTING"
        self.sentinel=get_random_bytes(16)
    def get_public_key(self):
        return (self.key.e, self.key.n)
    def get_ciphertext(self):
        return bytes_to_long(self.cipher.encrypt(self.flag_message.encode()))
    def get_pkcs_conforming(self, ciphertext,show=False):
        if isinstance(ciphertext,int):
            ciphertext=long_to_bytes(ciphertext)
        if isinstance(ciphertext,bytes):
            if len(ciphertext)>256:
                print ('Ciphertext too long')
                return None
            if len (ciphertext)<256:
                ciphertext=bytes([0]*(256-len(ciphertext)))+ciphertext
        result=self.cipher.decrypt(ciphertext,self.sentinel)
        return  result!=self.sentinel
    def get_pkcs_conforming_mult(self, ciphertexts,show=False):
        """Check several ciphertexts for PKCS conformity"""
        results=[]
        for ciphertext in ciphertexts:
            if isinstance(ciphertext,int):
                ciphertext=long_to_bytes(ciphertext)
            if isinstance(ciphertext,bytes):
                if len(ciphertext)>256:
                    print ('Ciphertext too long')
                    return None
                if len (ciphertext)<256:
                    ciphertext=bytes([0]*(256-len(ciphertext)))+ciphertext

            result=self.cipher.decrypt(ciphertext,self.sentinel)
            results.append(result!=self.sentinel)
        return results

In [4]:
vs=LocalVulnServerClient()
(e,N)=vs.get_public_key()
c=vs.get_ciphertext()
print(c)
print (vs.get_pkcs_conforming(c,False))
print (vs.get_pkcs_conforming(2))

# Your code goes here

13465018302617919138302626232318647050522989930094588118818758478862644872728883448382698314544858583522542682694675767990363419371205359678144006261769410411469448203931894514969804115808970208302215023417004111860674470931713693805285248212118284667095606822520161185367107564522349522447204075697719397703349872698656689382431419822514591127612311841891752241425696101815259490380699515152186432146734346249258525430477217594160016979561611452131399040945127631131054196658457260016882589028216661476731348711895243866952826138751933418601443112240592675879533226274283912996324396706036642818343441434547544486902
True
False


1. Инициализация параметров

In [5]:
# Init params
def init_attack(N):
    k = (N.bit_length() + 7) // 8
    B = 2**(8*(k - 2))
    return (k, B)

k, B = init_attack(N)
print(f"k: {k}\nB: {B}")

k: 256
B: 493118378773666493236005808848113280646424906459281677736363913383860094282041792193560812553755393427867400526762359916597283312232832658311281622107670335702985799671951234310153163915857728680359766210694390385082889078409114931668672093787783362893396695740300064741326536430985501229973638902647863548613194784388249853831252667031319724958132568898411896638150110768600863536200871492771279798342546336760614070411100118371556871830774626226863061725361438464769373851178286891558183314925099540247780495920664946518646198552749613009880449926596639031121858756000207590413184793166384097191709192063287296


2. Blinding

In [6]:
def blinding_step(oracle, e, N, c, B):
    s_0 = 1
    if oracle.get_pkcs_conforming(c,False):
        return (s_0, c, [(2*B, 3*B - 1)])

    while True:
        random_bytes = get_random_bytes(16)
        s_candidate = bytes_to_long(random_bytes) % N

        if s_candidate == 0:
            continue

        c_candidate = (c * pow(s_candidate, e, N)) % N
        if oracle.get_pkcs_conforming(c_candidate, False):
            return (s_candidate, c_candidate, [(2*B, 3*B - 1)])

s_0, c_0, interval = blinding_step(vs, e, N, c, B)
print(f"s_0: {s_0}\nc_0: {c_0}\ninterval: {interval}")

s_0: 1
c_0: 13465018302617919138302626232318647050522989930094588118818758478862644872728883448382698314544858583522542682694675767990363419371205359678144006261769410411469448203931894514969804115808970208302215023417004111860674470931713693805285248212118284667095606822520161185367107564522349522447204075697719397703349872698656689382431419822514591127612311841891752241425696101815259490380699515152186432146734346249258525430477217594160016979561611452131399040945127631131054196658457260016882589028216661476731348711895243866952826138751933418601443112240592675879533226274283912996324396706036642818343441434547544486902
interval: [(9862367575473329864720116176962265612928498129185633554727278267677201885640835843871216251075107868557348010535247198331945666244656653166225632442153406714059715993439024686203063278317154573607195324213887807701657781568182298633373441875755667257867933914806001294826530728619710024599472778052957270972263895687764997076625053340626394499162651377968237

2.a     Поиск первого $s$

In [12]:
def find_first_s(oracle, e, N, c_0, B):

    s_start = (N + 3*B - 1) // (3*B)
    s = s_start

    while True:
        candidates = list(range(s, s + 10000))
        ciphertexts = [(c_0 * pow(s_val, e, N)) % N for s_val in candidates]
        results = oracle.get_pkcs_conforming_mult(ciphertexts, False)

        for s_val, valid in zip(candidates, results):
            if valid:
                return s_val

        print(f"s: {s}")
        s += 10000

step_2a_first_s = find_first_s(vs, e, N, c_0, B)
print(f"s_1: {step_2a_first_s}")

s: 16485
s: 26485
s: 36485
s: 46485
s: 56485
s: 66485
s: 76485
s: 86485
s: 96485
s: 106485
s: 116485
s: 126485
s: 136485
s: 146485
s: 156485
s: 166485
s: 176485
s: 186485
s: 196485
s: 206485
s: 216485
s: 226485
s_1: 241470


2.b Поиск при нескольких интервалах

In [13]:
def multiple_intervals_search(oracle, e, N, c_0, s_prev):
    s = s_prev + 1
    while True:

        candidates = list(range(s, s + 1000))
        ciphertexts = [(c_0 * pow(s_val, e, N)) % N for s_val in candidates]
        results = oracle.get_pkcs_conforming_mult(ciphertexts, False)

        for s_val, valid in zip(candidates, results):
            if valid:
                return s_val
        s += 1000

2.c Поиск при одном интервале

In [14]:
def one_interval_search(oracle, e, N, c_0, s_prev, a, b, B):
    r = (2 * (b * s_prev - 2 * B) + N - 1) // N
    while True:
        s_min = (2 * B + r * N + b - 1) // b
        s_max = (3 * B - 1 + r * N) // a

        if s_min <= s_max:

            candidates = list(range(s_min, min(s_max + 1, s_min + 1000)))
            ciphertexts = [(c_0 * pow(s_val, e, N)) % N for s_val in candidates]
            results = oracle.get_pkcs_conforming_mult(ciphertexts, False)
            for s_val, valid in zip(candidates, results):
                if valid:
                    return s_val

        r += 1

3. Сужение интервалов

In [15]:
def narrow(M_prev, s_i, B, N):
    M_new = []

    for a, b in M_prev:
        r_min = (a * s_i - 3 * B + 1 + N - 1) // N
        r_max = (b * s_i - 2 * B) // N

        if r_min > r_max:
            continue

        for r in range(r_min, r_max + 1):
            new_a = max(a, (2 * B + r * N + s_i - 1) // s_i)
            new_b = min(b, (3 * B - 1 + r * N) // s_i)

            if new_a <= new_b:
                M_new.append((new_a, new_b))

    M_new.sort()
    merged = []
    for interval in M_new:
        if merged and interval[0] <= merged[-1][1] + 1:
            merged[-1] = (merged[-1][0], max(merged[-1][1], interval[1]))
        else:
            merged.append(interval)

    return merged

4. Проверка решения при одном интервале

In [16]:
def check(M, s_0, N):
    if M[0][0] == M[0][1]:
        m = (M[0][0] * inverse(s_0, N)) % N
        return long_to_bytes(m)
    else:
        return None

Реализация атаки (собираем финальное решение):

In [17]:
def bleichenbacher_attack(oracle, e, N, c):

    k, B = init_attack(N)
    print(f"k: {k}, B: {B}")

    s_0, c_0, M = blinding_step(oracle, e, N, c, B)
    print(f"s_0: {s_0}")
    print(f"Starting interval: {M}")

    i = 1
    s_prev = s_0

    while True:
        print(f"\nIter {i}, num_intervals: {len(M)}")

        if i == 1:
            s_i = find_first_s(oracle, e, N, c_0, B)
        elif len(M) >= 2:
            s_i = multiple_intervals_search(oracle, e, N, c_0, s_prev)
        else:
            a, b = M[0]
            s_i = one_interval_search(oracle, e, N, c_0, s_prev, a, b, B)

        print(f"found s_i = {s_i}")

        M = narrow(M, s_i, B, N)

        result = check(M, s_0, N)
        if result is not None:
            if result[0:2] == b'\x00\x02':
                sep_idx = result.find(b'\x00', 2)
                if sep_idx != -1:
                    plaintext = result[sep_idx + 1:]
                    return plaintext
            return result

        s_prev = s_i
        i += 1

message = bleichenbacher_attack(vs, e, N, c)

Выходные данные были обрезаны до нескольких последних строк (5000).
Iter 346, num_intervals: 1
found s_i = 13335433657290361052067909909984858803289208243770505756784880968752478038918320878514428461125752768685975410

Iter 347, num_intervals: 1
found s_i = 26670867314580722104135819819969717606578416487541011513569761937504956077836641757028856922251505537372091676

Iter 348, num_intervals: 1
found s_i = 53341734629161444208271639639939435213156832975082023027139523875009912155673283514057713844503011074744243718

Iter 349, num_intervals: 1
found s_i = 106683469258322888416543279279878870426313665950164046054279047750019824311346567028115427689006022149488527680

Iter 350, num_intervals: 1
found s_i = 213366938516645776833086558559757740852627331900328092108558095500039648622693134056230855378012044298977135849

Iter 351, num_intervals: 1
found s_i = 426733877033291553666173117119515481705254663800656184217116191000079297245386268112461710756024088597954332064

Iter 352, num_intervals

In [18]:
# message
print(f"message: {message}")

message: b'\x02u%\x85\x11X\xf7\x96\xa2z\x83L\xd3\xe4\xa7\xab\x90\xa3R\x1c\xcbjv\n\x0cr\x0e\xb7\xb5t\xff/\xaeP\xe0\xd2\xb6a\xcc/\x98\xb9\xca\xffk\xbc\xba\x87\xf8\xa5\x9e\x7f\xee\x81\nh#\x81\xc0\xbf\xb2\xcd\x86\xcf\x04\xde\xa5\x07\xbb\xa5h)\xb6J\xcc \xda{\xc7\x0c\r\x0f\xd33|\x7f\xe7Fn\x88\x88{C\xf9\xb7F)-\xf3\xda\x9c\xa4a\x89X\x9c\xc1\x8eZI\xd8i\x10\x103\xab\xa8\xd975\xff\xd1\xeexc\xcbc\xd0\x08\xfb\x8a\x80\xdaJ\x9b\xd7sbLjkh\x82x\xf6\n\x07\xf6\xfe\xaem\x14\x19\xf5\xc0\xc3\x94M\x92T\x01\xa8\xcf4\x15\x9a\x1d\x08\xbce\x19\xd2\xd2s\x184m\xe4\x19\xb9\xbf$\x91\xf9\xea\x1f\x14V=\xc7\xeb\xbd\xebzw @\x1fn\xecf\xca$CT\xcd\xf9Y\x10\xfe\xf8\xee\xab\x9bZ\x95\x03\x8e~\x8b\xf6M\x8fM\x14\xff\xb5PX\xbb\x94\ru\xb4\xc0V;|\x00FLAG FOR TESTING'


Теперь, когда вы сломали локальный сервер, нужно просто сделать то же самое для удаленного

In [19]:
import socket
import re
from Crypto.Util.number import inverse, long_to_bytes, bytes_to_long
class VulnServerClient:
    def __init__(self,show=True):
        """Инициализация, подключаемся к серверу"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1344))
        if show:
            print (self.recv_until().decode())
    def recv_until(self,symb=b'\n>'):
        """Получаем сообщения с сервера, по умолчанию до первого приглашения"""
        data=b''
        while True:

            data+=self.s.recv(1)
            if data[-len(symb):]==symb:
                break
        return data
    def get_public_key(self,show=True):
        """Получаем открытый ключ с сервера"""
        self.s.sendall('public\n'.encode())
        response=self.recv_until().decode()
        if show:
            print (response)
        e=int(re.search('(?<=e: )\d+',response).group(0))
        N=int(re.search('(?<=N: )\d+',response).group(0))
        return (e,N)

    def get_ciphertext(self,show=True):
        """Получаем шифротекст с сервера"""
        self.s.sendall('ciphertext\n'.encode())
        response=self.recv_until().decode()
        if show:
            print (response)
        c=bytes_to_long(bytes.fromhex(re.search('(?<=ciphertext: )[0-9a-f]+',response).group(0)))
        return c

    def get_pkcs_conforming(self, ciphertext, show=True):
        """Проверяем шифротекст на соответствие PKCS"""
        if isinstance(ciphertext,int):
            ciphertext=long_to_bytes(ciphertext)
        if len(ciphertext)>256:
            print ('Ciphertext too long')
            return None
        if len (ciphertext)<256:
            ciphertext=bytes([0]*(256-len(ciphertext)))+ciphertext
        ciphertext_hex=ciphertext.hex()

        self.s.sendall(f'unpad {ciphertext_hex}\n'.encode())

        response=self.recv_until().decode()
        if show:
            print (response)
        return response.find('error')==-1

    def get_pkcs_conforming_mult(self, ciphertexts,show=False):
        """Проверяем несколько шифротекстов на соответствие PKCS"""
        newc=[]
        for x in ciphertexts:
            if isinstance(x,int):
                ciphertext=long_to_bytes(x)
            else:
                ciphertext=x
            if len(ciphertext)>256:
                print ('Ciphertext too long')
                return None
            if len (ciphertext)<256:
                ciphertext=bytes([0]*(256-len(ciphertext)))+ciphertext
            ciphertext_hex=ciphertext.hex()
            newc.append(ciphertext_hex)
        self.s.sendall(('\n'.join([f'unpad {chex}' for chex in newc])+'\n').encode())

        results=[]
        for i in range(len(newc)):
            response=self.recv_until().decode()
            if show:
                print (response)
            results.append(response.find('error')==-1)
        return results



    def __del__(self):
        self.s.close()

vs=VulnServerClient()
(e,N)=vs.get_public_key()
c=vs.get_ciphertext()
print (vs.get_pkcs_conforming(c, False))

# Your code goes here
message_vc = bleichenbacher_attack(vs, e, N, c)

<>:26: SyntaxWarning: invalid escape sequence '\d'
<>:27: SyntaxWarning: invalid escape sequence '\d'
<>:26: SyntaxWarning: invalid escape sequence '\d'
<>:27: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-520/3578191976.py:26: SyntaxWarning: invalid escape sequence '\d'
  e=int(re.search('(?<=e: )\d+',response).group(0))
/tmp/ipython-input-520/3578191976.py:27: SyntaxWarning: invalid escape sequence '\d'
  N=int(re.search('(?<=N: )\d+',response).group(0))


Выходные данные были обрезаны до нескольких последних строк (5000).
Iter 350, num_intervals: 1
found s_i = 158811585161500976264929441885247789744298609699475241693164463346098171731321348227869393975263679661863172236

Iter 351, num_intervals: 1
found s_i = 317623170323001952529858883770495579488597219398950483386328926692196343462642696455738787950527359323726433431

Iter 352, num_intervals: 1
found s_i = 635246340646003905059717767540991158977194438797900966772657853384392686925285392911477575901054718647452926168

Iter 353, num_intervals: 1
found s_i = 1270492681292007810119435535081982317954388877595801933545315706768785373850570785822955151802109437294905881988

Iter 354, num_intervals: 1
found s_i = 2540985362584015620238871070163964635908777755191603867090631413537570747701141571645910303604218874589811852935

Iter 355, num_intervals: 1
found s_i = 5081970725168031240477742140327929271817555510383207734181262827075141495402283143291820607208437749179623913442

Iter 356, num_int

In [20]:
print(f"message: {message_vc}")

message: b"\x02\x17\xdd\xb6\x9e\xfe\x89)\x8d\x96\xfa\xf2w\xda\xd8(\xfa\xf1\x08\x1d\x07)\t\rj-\xf0\xc6\xab\xd7D\xdb\xacCO\x80\x15\xc5P\x1bZw\xf8\xea\xd2\xfbe$\xa6\x8b\xb4\x8c5t\xe8S\xbd=\x9e\xae-\x16\xc9\xf0bU%\xf1\x16>\xe1\xda/s\x85\xb6\x1e\xb0\x1a\x98\x9ax1\xb8Q*'\\\xf4\xe4\xdb_\x8b\xdb\xd6Pd\xfb:\xf0Fu~\xc4\xc2\xc5\x1b \xfd\xae\xb1n,>\x9b\xb0\xbb\x1e\x8e\xdd\x9c\xeb\xa4\x06w\x82dWY\x08\xa1\x16\xef\xa3\xc8\xf6Y\xe3\x99\xaf\x08\x061\x11\x930,9F\x83\xddn\x06\xc5\xa6\xb6\x1fKh\x93\xa8\x97 \x99\xbf\xa1\xe7\xf94\xbdfS\xb5\xa6\xe0\x92\xfa\xd2\xb07\x88\xa6\x9b%\x1d\xeb_?\xb1\x18\x10:\xd8\x00CRYPTOTRAINING{c0ngr475_0n_d01ng_y0ur_f1r57_r34lw0rld_4tt4ck}"
